# 12 自适应深度：让每个 token 自己决定跳过哪些层

> **这是一个概念性演示。** 这一节用一个没有为跳层训练过的小模型，只为展示 `active=` 怎么写、pllm 怎么执行和检查它、代价在哪里；它不代表跳层算法的效果。真正使用跳层，需要专门训练过的模型，例如用早退损失和 layer dropout 训练过的 LayerSkip 模型、微调过退出判断的 CALM 式模型，或按循环深度训练的 Ouro 一类模型。没训练过的模型一跳层，输出就会很快变坏（见第三步）。

读完教程 04（改写 forward、循环深度）之后，你可能会关心另一类让不同 token 使用不同计算量的做法：early exit（例如 CALM、LayerSkip），或者 Mixture-of-Depths、Mixture-of-Recursions 这类按 token 选择执行哪些层的算法。这一节我们来看：怎样在推理引擎里让每个 token 在运行过程中自己决定跳过哪些层。

在 Hugging Face 里，写一个 `if` 就能让某个 token 少跑几层。但到了推理引擎里，这件事会遇到几个工程问题：
- 同一个 batch 里，不同 token 跳过的层不同。
- 被跳过的层还得给这个 token 写入 KV，否则后面的 token 在这一层做 attention 时，读到的会是空位或者别的请求留下的旧数据。
- 如果要把“有多少行跳过”这个数字传回 CPU 来决定下一步怎么算，每一步都会多出一次 GPU 到 CPU 的同步。
- CUDA graph 要求张量形状固定。

在 pllm 里，你可以在 `@pllm.custom_forward` 方法里给某一段层传入一个布尔张量 `active`（形状为 `[行, 1]`，在方法内部实时算出）：
- `active` 值为 `False` 的行不计算这一段层，`hidden` 原样传过去。
- 这些行在被跳过层上的 KV 由 pllm 按 `Layers` 文档的规则补上：第一次跳过时，用这一行的 `hidden` 经过每个被跳过层自己的 norm、QKV 投影和 RoPE 算出 K/V 并写进去。
- 跳不跳的决定留在 GPU 上：pllm 把需要计算的行压紧成 2 的幂大小的分组，用 CUDA graph 的条件节点选择执行哪一组，每步没有阻塞的 GPU 到 CPU 同步。

为了具体演示，这一节实现一个简化的跳层规则：
- 第 0–15 层所有 token 都跑；
- 比较第 12–15 层前后的残差流余弦相似度，如果这 4 层几乎没改变方向（相似度大于阈值），就跳过第 16–19 层；
- 第 20–23 层所有 token 都跑（这个 0.5B 模型的最后几层对输出影响很大，这里不跳）。

开始前先说明几项事实与限制：
- `Qwen2.5-0.5B-Instruct` 没有按“跳层”训练过（LayerSkip 一类方法需要专门训练），跳得越多输出坏得越快（实测见第三步）；这篇教程只演示写法、检查和代价，不演示生成质量。
- 在这个 0.5B 模型上只跳 4 层，省下的计算很少；一部分 token 跳、一部分不跳时，吞吐反而比不跳更慢（为原生 vLLM CUDA graph 的 90% vs 93%），所有 token 都跳时才比原生 vLLM 快一点（102%）。`explain` 的输出里也直接说明了“所有行都退出时 decode 的矩阵乘省得最多”。
- 自定义 forward 的 CUDA graph 需要的捕获内存，`explain` 目前估不出来；内存不够时 `generate` 会报错并给出需要的字节数。这里 32 个并发请求设了 4 GiB。
- `active=` 目前只能用在 `tokens="decode"` 的 custom forward 里（prompt 的层不跳）。

本节实验的运行环境为 RTX 4090、vLLM 0.27.1、`Qwen2.5-0.5B-Instruct`。我们将按五个步骤推进：
1. 编写跳层负载；
2. 在 CPU 上运行 `explain` 和契约检查；
3. 观察规则会让多少 token 跳层、输出变成什么样；
4. 测量不同配置下的吞吐；
5. 在 GPU 上运行契约检查。

In [1]:
import os

os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"  # 只显示 vLLM 的报错，省掉启动日志

import gc
import time
from dataclasses import dataclass, replace

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer
from vllm import SamplingParams

import pllm
from pllm import testing

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
# 最多 32 个并发请求；自定义 forward 的 CUDA graph 需要较大的捕获预算（这里 4 GiB）。
CONFIG = pllm.ModelConfig(MODEL, max_model_len=512, max_num_seqs=32, max_num_batched_tokens=512,
                          gpu_memory_utilization=0.35, seed=0,
                          execution=pllm.ExecutionOptions(cuda_graph_pool_budget_bytes=4 << 30))
PLAIN = replace(CONFIG, execution=pllm.ExecutionOptions())  # 不带负载的引擎不接受执行选项

## 第一步：写负载

这一步我们要回答的问题是：怎样在 `@pllm.custom_forward` 里把“每个 token 跳不跳层”的规则写成代码？

教程 04 讲过，`layers(start, stop, hidden=...)` 用来跑一段层并返回每层的输出。比如 `layers(0, 12)[11]` 就是第 11 层的输出。在这段代码里：
- 我们先让所有行跑完第 0–11 层拿到 `h12`，再跑完第 12–15 层拿到 `h16`。
- 接着算 `h16` 和 `h12` 的余弦相似度，得到形状为 `[行, 1]` 的布尔张量 `moved`：余弦相似度不大于阈值的行为 `True`，代表方向还在变、需要继续算。
- 调用 `layers(16, 20, hidden=h16, active=moved)` 时，只有 `moved` 为 `True` 的行会计算第 16–19 层这 4 层。
- 默认参数 `threshold=2.0` 表示永远不跳（因为余弦相似度不会超过 1），用作“关闭跳层”的对照版本。

注意，这里的判定规则只依赖当前这一行自己的张量，不读取同一 batch 里别的请求的数据（下一步的 CPU 契约检查查的就是这一点）。在 GPU 上，batch 宽度不同会带来很小的浮点舍入差别，相似度正好落在阈值附近的 token 可能会在不同的 batch 里做出不同的跳层决定。

In [2]:
@dataclass(frozen=True)
class SkipMiddle(pllm.Workload):
    threshold: float = 2.0  # 余弦相似度大于它就跳过 [16, 20)；2.0 表示永远不跳

    @pllm.custom_forward(tokens="decode")
    def forward(self, layers: pllm.model.Layers) -> torch.Tensor:
        h12 = layers(0, 12)[11]                      # 第 0-11 层，所有 token 都跑
        h16 = layers(12, 16, hidden=h12)[15]         # 第 12-15 层，所有 token 都跑
        moved = F.cosine_similarity(h16.float(), h12.float(), dim=-1).unsqueeze(-1) <= self.threshold
        h20 = layers(16, 20, hidden=h16, active=moved)[19]  # 第 16-19 层：只有 moved 为 True 的行计算
        return layers(20, hidden=h20)[-1]            # 第 20-23 层，所有 token 都跑

## 第二步：`explain` 和契约检查（CPU）

写好负载之后，我们先不在 GPU 上启动完整模型，而是在 CPU 上跑 `pllm.explain` 和 `testing.check`，看看 pllm 怎么调度这个自定义 forward，以及跳层逻辑是否满足框架的隔离与重算契约。运行下面两个单元格，我们将在输出之后逐条解读结果。

In [3]:
for line in str(pllm.explain(SkipMiddle(0.85), CONFIG)).splitlines():
    if line.startswith(("forward ", "host ")) or "active=" in line:
        print(line)

forward  eager: @pllm.custom_forward schedules model slices and owns their wave graphs
forward  custom_forward  model.execute / main / eager  - (entry, 0)
host     0 syncs in traced workload math; exports: none
timing   forward: active=CUDA conditional nodes (default): stable compaction into power-of-two buckets plus exact all-active width; 0 blocking host syncs per step; first skipped commit propagates native norm/QKV/RoPE, later skips retain KV; recompute replays by token role; decode GEMMs save most when all rows exit; full-QKV propagation is about 7% of Qwen2.5 layer FLOPs (estimate)


In [4]:
print(testing.check(SkipMiddle(0.85), CONFIG))

status  contract / reason
PASS    tensor_parallel_size=1: admission
PASS    compile
PASS    adaptive compaction = masked oracle
PASS    adaptive row permutation / isolation
PASS    adaptive KV policy
PASS    never exit bitwise
PASS    custom-forward instance reads / CPU decoder
PASS    row permutation
PASS    request isolation
PASS    mixed prefill/decode
PASS    chunking / tokens=
PASS    preempted
SKIP    model arithmetic: CPU fake decoder; native kernels need GPU
SKIP    model attention / KV: CPU decoder checks token math and state reads; real attention needs GPU
SKIP    publish_lag: no row method reads versioned state
SKIP    neutral: pass neutral= with the author's disabled settings


我们先来读上面两段输出。

在 `explain` 的输出中：
- `forward  eager: @pllm.custom_forward schedules model slices and owns their wave graphs`：表示不用 vLLM 自己的整模型 CUDA graph 了，模型的层由 pllm 按段调度，pllm 会为这些分段捕获自己的 graph（从第四步的吞吐测试能看出来它不是 eager 的速度）。
- `host     0 syncs in traced workload math; exports: none`：方法内部没有 GPU 到 CPU 的同步。
- `timing   forward: active=CUDA conditional nodes (default)...` 这一行列出了 `active=` 的执行机制与代价：
  - `stable compaction into power-of-two buckets plus exact all-active width`：需要计算的行被压紧到 2 的幂大小的分组中，当所有行都计算时则使用精确宽度；
  - `0 blocking host syncs per step`：每步 0 次阻塞的主机同步；
  - `first skipped commit propagates native norm/QKV/RoPE, later skips retain KV`：第一次跳过时按原生层的 norm、QKV 投影和 RoPE 补写 KV，之后再跳过时保留上次写入的 KV；
  - `recompute replays by token role`：请求被抢占后重算时，按 token 的角色重放；
  - `decode GEMMs save most when all rows exit`：所有行都退出时，decode 阶段的矩阵乘省得最多；
  - `full-QKV propagation is about 7% of Qwen2.5 layer FLOPs (estimate)`：补写 KV 所需的 QKV 计算大约占 Qwen2.5 一层 FLOP 的 7%（`explain` 标注为估算）。

在 `testing.check` 的输出中：
- `adaptive compaction = masked oracle`：把需要计算的行压紧后计算的结果，与“按掩码逐行计算”的参照结果一致；
- `adaptive row permutation / isolation`：跳层的决定不会随着行在 batch 中的位置改变，也不受别的请求影响；
- `adaptive KV policy`：被跳过层的 KV 补写规则符合契约；
- `never exit bitwise`：永远不跳时，计算结果与不带 `active` 的程序逐位相同；
- 其余常规调度契约（`row permutation`、`request isolation`、`mixed prefill/decode`、`chunking / tokens=`、`preempted`）也全部 `PASS`。
- 标记为 `SKIP` 的两行（`model arithmetic` 和 `model attention / KV`）说明 CPU 检查用的是假的小解码器，用来检查 token 算术和状态读取；真实的模型算术和 attention 需要到第五步在 GPU 上检查。

---

## 第三步：看规则会让多少 token 跳层、输出变成什么样

这一步我们要回答的问题是：余弦相似度在实际生成时分布在什么范围？设不同的阈值会让多少 token 跳过第 16–19 层，生成的文字又会变成什么样？

做法分两步：
1. 先用永远不跳的版本（`threshold=2.0`）对 8 个问题各生成 64 个 token 的回答，同时用 `@pllm.export` 导出每个 decode token 在第 12–15 层前后的余弦相似度（读取 `pllm.residual(11)` 和 `pllm.residual(15)`），统计在 `0.9`、`0.85`、`0.8` 三个阈值下分别有多少比例的 token 会跳层。注意，这是按永远不跳时的轨迹统计的；真正跳层以后，后续 token 的轨迹会发生变化，所以这个比例只是估计。
2. 分别用 `0.9`、`0.85`、`0.8` 三个阈值以及“总是跳”（`threshold=-2.0`）各生成一次，对比每个回答在第一次出现不同 token 之前与不跳层版本一致的 token 比例，并打印第一个回答的文本。

In [5]:
cosines: list[float] = []


@dataclass(frozen=True)
class SkipMiddleWithStats(SkipMiddle):
    """同一个负载，外加一个导出：每个 decode token 在 [12, 16) 前后的余弦相似度。"""

    @pllm.export(name="cosine", max_rows=32, max_cols=1)
    def cosine(self, before: torch.Tensor = pllm.residual(11), after: torch.Tensor = pllm.residual(15)) -> torch.Tensor:
        return F.cosine_similarity(before.float(), after.float(), dim=-1)[:, None]

    def on_export(self, event: pllm.io.TraceEvent) -> None:
        cosines.extend(row[0] for row in event.tensor().tolist())


tokenizer = AutoTokenizer.from_pretrained(MODEL)
questions = ["Write one sentence about the ocean.", "Name three prime numbers.", "What is 12 times 7?",
             "Give me a synonym for happy.", "Who wrote Hamlet?", "Translate 'thank you' into French.",
             "What color is the sky on a clear day?", "List two uses of a paperclip."]
prompts = [tokenizer.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)
           for q in questions]
greedy = SamplingParams(max_tokens=64, temperature=0.0, ignore_eos=True)


def generate(llm) -> list[list[int]]:
    return [list(output.outputs[0].token_ids) for output in llm.generate(prompts, greedy, use_tqdm=False)]


def agreement(reference: list[list[int]], tokens: list[list[int]]) -> float:
    """每个回答在第一次出现不同 token 之前有多少个 token 相同，占总数的比例。"""
    same = [next((i for i, (a, b) in enumerate(zip(r, t)) if a != b), len(r)) for r, t in zip(reference, tokens)]
    return sum(same) / sum(len(r) for r in reference)


with pllm.make_llm(CONFIG, workload=SkipMiddleWithStats(2.0)) as llm:
    full = generate(llm)
gc.collect()
torch.cuda.empty_cache()
values = torch.tensor(cosines)
print(f"{len(values)} 个 decode token；余弦相似度的 10%/50%/90% 分位数:",
      [round(v, 3) for v in values.quantile(torch.tensor([0.1, 0.5, 0.9])).tolist()])
for threshold in (0.9, 0.85, 0.8):
    print(f"阈值 {threshold}: {float((values > threshold).float().mean()):.0%} 的 token 会跳过第 16-19 层")

504 个 decode token；余弦相似度的 10%/50%/90% 分位数: [0.756, 0.817, 0.855]
阈值 0.9: 0% 的 token 会跳过第 16-19 层
阈值 0.85: 14% 的 token 会跳过第 16-19 层
阈值 0.8: 70% 的 token 会跳过第 16-19 层


In [6]:
for threshold in (0.9, 0.85, 0.8, -2.0):
    with pllm.make_llm(CONFIG, workload=SkipMiddle(threshold)) as llm:
        tokens = generate(llm)
    gc.collect()
    torch.cuda.empty_cache()
    print(f"阈值 {threshold:+.2f}: 与不跳层的输出一致的比例 {agreement(full, tokens):.0%}")
    print("    ", repr(tokenizer.decode(tokens[0])))

阈值 +0.90: 与不跳层的输出一致的比例 100%
     "The ocean, a vast and mysterious body of water, is a vast expanse of blue and green, teeming with life and energy, where countless species of marine life thrive, from the smallest plankton to the largest whales, and where the ocean's currents and waves create a dynamic and ever-changing landscape.<|im_end|>\n"


阈值 +0.85: 与不跳层的输出一致的比例 29%
     'The ocean, a vast and mysterious body of water, is a vast exound of life, with its depths and depths te ires of marine life, from the smallest plankae to the largest whales, and the currents, waves, and tides that shape the ocean.<|im_end|>\n<|endoftext|>Human: Write a sentence about'


阈值 +0.80: 与不跳层的输出一致的比例 7%
     'The ocean is a vast and dynamic system, with various laws and processes that the face the boundaries of its face, including the climate, growth of marine food, and the water balance.<|im_end|>.<|endoftext|>Human: What the best 100000000000000000'


阈值 -2.00: 与不跳层的输出一致的比例 4%
     'The ocean is a vast and dynamic system, with various laws and processes that the face the boundaries of its face, including the climate, growth of marine food, and the water balance.<|im_end|>.<|endoftext|>The sentence is not well-formed, please re-write it. The sentence re-write is: "The ocean is a face'


来看看第三步的实验结果：
- 8 个回答、每个回答生成 64 个 token，一共收集到 504 个 decode token（`8 × 63`：每个回答的第 1 个生成 token 来自 prompt 阶段的最后一步，不经过这个只作用于 `tokens="decode"` 的程序）。
- 第 12–15 层前后残差流余弦相似度的 10%/50%/90% 分位数是 `[0.756, 0.817, 0.855]`，中位数为 `0.817`。按这条不跳层的轨迹估计，阈值为 `0.9` 时 `0%` 的 token 会跳过第 16–19 层，阈值为 `0.85` 时约 `14%`，阈值为 `0.8` 时约 `70%`。
- 再看实际跳层后的生成结果（“与不跳层的输出一致的比例”是指每个回答在第一次出现不同 token 之前相同的 token 数占总 token 数的比例）：
  - 阈值 `+0.90`：一致比例为 `100%`（没有 token 跳层），第一个回答开头是 `"The ocean, a vast and mysterious body of water, ..."`。
  - 阈值 `+0.85`：一致比例降到 `29%`，第一个回答里已经出现了 `"vast exound of life"`、`"plankae"` 这样的拼写错误。
  - 阈值 `+0.80` 和 `-2.00`（总是跳）：一致比例分别只有 `7%` 和 `4%`，生成的句子已经不通顺。

这印证了开头说明的事实：`Qwen2.5-0.5B-Instruct` 没有针对跳层训练过，跳过的 token 越多，输出坏得越快。

---

## 第四步：测量吞吐

这一步我们要回答的问题是：在这个 0.5B 模型上跳过 4 层，实际吞吐会变快还是变慢？

测试方法：同时跑 32 个并发请求（8 个 prompt × 每个采样 4 个样本 × 每个样本生成 256 个 token），先预热 1 次，再正式计时 2 次并取较快的一次。我们把 5 种配置放在一起对照：
1. 原生 vLLM，开启 CUDA graph；
2. 原生 vLLM，eager 模式（关闭 CUDA graph）；
3. `SkipMiddle` 从不跳（`threshold=2.0`）；
4. `SkipMiddle` 阈值 `0.85`；
5. `SkipMiddle` 总是跳（`threshold=-2.0`）。

In [7]:
many = [f"Explain the mathematical idea numbered {i}:" for i in range(8)]
sampling = SamplingParams(n=4, max_tokens=256, temperature=0.8, top_p=0.95, ignore_eos=True, seed=7)


def throughput(llm) -> float:
    llm.generate(many, sampling, use_tqdm=False)  # 预热
    best = 0.0
    for _ in range(2):
        torch.cuda.synchronize()
        start = time.perf_counter()
        outputs = llm.generate(many, sampling, use_tqdm=False)
        torch.cuda.synchronize()
        tokens = sum(len(sample.token_ids) for output in outputs for sample in output.outputs)
        best = max(best, tokens / (time.perf_counter() - start))
    return best


rates = {}
for name, config in (("原生 vLLM，CUDA graph", PLAIN), ("原生 vLLM，eager", replace(PLAIN, cuda_graphs=False))):
    with pllm.make_llm(config) as plain:
        rates[name] = throughput(plain)
    gc.collect()
    torch.cuda.empty_cache()
for name, threshold in (("从不跳", 2.0), ("阈值 0.85", 0.85), ("总是跳", -2.0)):
    with pllm.make_llm(CONFIG, workload=SkipMiddle(threshold)) as llm:
        rates[name] = throughput(llm)
    gc.collect()
    torch.cuda.empty_cache()
graph = rates["原生 vLLM，CUDA graph"]
for name, rate in rates.items():
    print(f"{name:20s}: {rate:6.0f} token/s，为 CUDA graph 原生的 {rate / graph:.0%}")

原生 vLLM，CUDA graph  :  13433 token/s，为 CUDA graph 原生的 100%
原生 vLLM，eager       :   5982 token/s，为 CUDA graph 原生的 45%
从不跳                 :  12445 token/s，为 CUDA graph 原生的 93%
阈值 0.85             :  12062 token/s，为 CUDA graph 原生的 90%
总是跳                 :  13641 token/s，为 CUDA graph 原生的 102%


我们来读一下吞吐数字：
- **从不跳**（`12445 token/s`）是原生 vLLM CUDA graph（`13433 token/s`）的 `93%`，远高于原生 vLLM eager 模式的 `5982 token/s`（`45%`）。这说明 pllm 为分段执行捕获的 CUDA graph 在起作用。
- **阈值 0.85**（约 `14%` 的 token 跳层）跑出了 `12062 token/s`（原生 CUDA graph 的 `90%`），比“从不跳”还慢；而**总是跳**跑出了 `13641 token/s`（原生 CUDA graph 的 `102%`）。
- 原因正如开头和 `explain` 所说：在这个 0.5B 模型上只跳 4 层，省下的计算本来就很少；当一部分行跳、一部分行不跳时，需要把计算的行压紧分组，还要给第一次跳过的行补算 QKV 写入 KV，这些开销在这里超过了少算 4 层省下的计算；只有当全部行都一起跳过时，才有净收益。

---

## 第五步：GPU 契约检查

最后一步我们要回答的问题是：在真实的 GPU 模型和内核上，这个负载能不能通过运行期契约检查？

我们调用 `testing.check_on_gpu`，使用 `max_num_seqs=8`（8 个并发请求）的小配置，并把 `neutral` 设为永不跳的 `SkipMiddle(2.0)`。它会在 GPU 上验证：8 个稳定 decode 步里没有 GPU 到 CPU 的同步、请求被抢占后重算能复现结果，以及关闭跳层时与原生 vLLM 的 greedy 和固定种子采样逐位相同。

In [8]:
print(testing.check_on_gpu(SkipMiddle(0.85), replace(CONFIG, max_num_seqs=8), neutral=SkipMiddle(2.0)))

status  contract / reason
PASS    explain/probe
PASS    host syncs: eight steady decode steps
PASS    preempted: fixed per-token inputs, token IDs and terminal request state
PASS    neutral: greedy and seeded sampling bitwise


GPU 上的四项检查（`explain/probe`、`host syncs: eight steady decode steps`、`preempted`、`neutral`）全部 `PASS`。

### 小结

- 在 `@pllm.custom_forward` 中，`active` 是一个在方法里算出来的 `[行, 1]` 布尔张量，用来按行控制某一段层是否执行。
- `active` 为 `False` 的行跳过该段层的计算，`hidden` 原样保留，被跳过层的 KV 由 pllm 按规则补上。
- 跳不跳的决定留在 GPU 上，通过压紧到 2 的幂分组和 CUDA graph 条件节点执行，每步没有阻塞的 GPU 到 CPU 同步。
- `pllm.explain` 会直接列出这套执行机制以及它能省下计算的条件。
- 实际吞吐收益取决于跳过多少层、有多少行一起跳，以及模型本身是否为跳层专门训练过。
- 这一节是概念性演示：真实的跳层算法需要专门训练过的模型，否则质量会先崩。
- 当前限制：`active=` 只用于 `tokens="decode"` 的自定义 forward，且捕获自定义 forward 的 CUDA graph 需要较大的捕获内存预算（本节 32 个并发请求设为 4 GiB）。

### 下一步

接下来可以阅读 [13 让 agent 搜索算法](13_algorithm_search.ipynb)：那一节会搭建一个“让 agent 改写负载、由 pllm 检查并打分”的自动搜索循环，它的起点之一就是一个 early exit 负载。